In [12]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
import warnings

from src.features import engineer_features
warnings.filterwarnings('ignore')

In [13]:
print("Loading data and calculating Historical Target Aggregations...")
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

train['geohash'] = train['geohash'].fillna('Missing')
test['geohash'] = test['geohash'].fillna('Missing')

# --- THE GRANDMASTER UPGRADE: Historical Target Mapping ---
# Calculate historical traffic stats for each geohash based STRICTLY on training data
geo_stats = train.groupby('geohash')['demand'].agg(
    geo_mean='mean', 
    geo_median='median', 
    geo_std='std'
).reset_index()

# Map these historical facts to both train and test sets
train = train.merge(geo_stats, on='geohash', how='left')
test = test.merge(geo_stats, on='geohash', how='left')

# Calculate historical traffic by Hour
train['hour_temp'] = train['timestamp'].apply(lambda x: int(str(x).split(':')[0]) if pd.notnull(x) else 0)
test['hour_temp'] = test['timestamp'].apply(lambda x: int(str(x).split(':')[0]) if pd.notnull(x) else 0)

hour_stats = train.groupby('hour_temp')['demand'].agg(hour_mean='mean').reset_index()
train = train.merge(hour_stats, on='hour_temp', how='left').drop(columns=['hour_temp'])
test = test.merge(hour_stats, on='hour_temp', how='left').drop(columns=['hour_temp'])

# Combine for final base engineering
test['demand'] = -1 
df = pd.concat([train, test], ignore_index=True)

Loading data and calculating Historical Target Aggregations...


In [14]:
print("Applying spatial features...")
df_engineered = engineer_features(df)

train_df = df_engineered[df_engineered['demand'] != -1].copy()
test_df = df_engineered[df_engineered['demand'] == -1].copy()

X = train_df.drop(columns=['Index', 'demand'])
y = train_df['demand']
X_test = test_df.drop(columns=['Index', 'demand'])

# Fill missing aggregations in the test set with global means to prevent NaNs
X_test['geo_mean'] = X_test['geo_mean'].fillna(X['geo_mean'].mean())
X_test['geo_median'] = X_test['geo_median'].fillna(X['geo_median'].median())
X_test['geo_std'] = X_test['geo_std'].fillna(X['geo_std'].mean())

cat_features = ['geohash', 'RoadType', 'Weather']

Applying spatial features...


In [15]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lgb_test_preds = np.zeros(len(X_test))
cat_test_preds = np.zeros(len(X_test))

print("Initiating Stable LightGBM & CatBoost Ensemble...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # --- LightGBM ---
    X_train_lgb, X_val_lgb, X_test_lgb = X_train.copy(), X_val.copy(), X_test.copy()
    for col in cat_features:
        X_train_lgb[col] = X_train_lgb[col].astype('category')
        X_val_lgb[col] = X_val_lgb[col].astype('category')
        X_test_lgb[col] = X_test_lgb[col].astype('category')

    # Higher number of estimators, more conservative learning rate
    lgb = LGBMRegressor(n_estimators=2500, learning_rate=0.02, random_state=42, n_jobs=-1)
    lgb.fit(
        X_train_lgb, y_train, eval_set=[(X_val_lgb, y_val)], 
        callbacks=[early_stopping(stopping_rounds=100), log_evaluation(period=0)]
    )
    lgb_test_preds += lgb.predict(X_test_lgb) / kf.n_splits
    
    # --- CatBoost ---
    cat = CatBoostRegressor(iterations=2500, learning_rate=0.03, cat_features=cat_features, random_seed=42, verbose=0)
    cat.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100)
    cat_test_preds += cat.predict(X_test) / kf.n_splits
    
    print(f"Fold {fold+1} Completed.")

# Robust 50/50 blend (prevents optimizer overfitting)
final_predictions = (lgb_test_preds * 0.5) + (cat_test_preds * 0.5)

# Safety clip
final_predictions = np.clip(final_predictions, a_min=0.0, a_max=None)

submission = pd.DataFrame({
    'Index': test_df['Index'].astype(int),
    'demand': final_predictions
})
submission.to_csv('submission.csv', index=False)
print("submission.csv generated successfully! Ready for max score upload.")

Initiating Stable LightGBM & CatBoost Ensemble...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.163925 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2459
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 17
[LightGBM] [Info] Start training from score 0.093784
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2500]	valid_0's l2: 0.000757783
Fold 1 Completed.
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, ma